# FD001 Dataset Exploration

This notebook examines the structure and measured behavior of NASA C-MAPSS FD001 before any target engineering or modeling. It does not generate training RUL targets, normalize model inputs, create sequence windows, or train a model.

## Files and row structure

- `train_FD001.txt`: complete run-to-failure trajectories for 100 training engines.
- `test_FD001.txt`: partial trajectories for a separate set of 100 test engines.
- `RUL_FD001.txt`: one true remaining-cycle count for the final observed row of each test engine.

A train/test row contains a unit identifier, its operational-cycle index, three operational settings, and 21 sensor measurements. Multiple chronologically ordered rows share a unit ID because an engine is observed over many cycles. `cycle` orders observations within a unit; it is not a physical sensor.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from turbofan_rul.data import SENSOR_COLUMNS, load_fd001
from turbofan_rul.eda import (
    column_profile,
    plot_engine_lifetimes,
    plot_operational_settings,
    plot_sensor_correlation,
    plot_sensor_distributions,
    plot_sensor_evolution,
    plot_sensor_trends,
    plot_sensor_variance,
    representative_engine_ids,
    sensor_trend_summary,
    split_summary,
    validate_split,
)

DATA_DIRECTORY = PROJECT_ROOT / "data" / "raw" / "CMAPSSData"
FIGURE_DIRECTORY = PROJECT_ROOT / "reports" / "figures"
pd.set_option("display.max_rows", 30)
pd.set_option("display.max_columns", 30)

## Load FD001

The reusable loader assigns explicit column names, requires exactly 26 numeric columns for train/test files, checks finite values, and gives IDs/cycles integer types. It does not alter or drop measurements.

In [2]:
fd001 = load_fd001(DATA_DIRECTORY)
train = fd001.train
test = fd001.test
test_rul = fd001.test_rul

print(f"Training shape: {train.shape}")
print(f"Test shape:     {test.shape}")
print(f"Test RUL shape: {test_rul.shape}")

Training shape: (20631, 26)
Test shape:     (13096, 26)
Test RUL shape: (100, 2)


In [3]:
display(train.head())

,unit_id,cycle,operational_setting_1,operational_setting_2,operational_setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,sensor_6,sensor_7,sensor_8,sensor_9,sensor_10,sensor_11,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,21.61,554.36,2388.06,9046.19,1.3,47.47,521.66,2388.02,8138.62,8.4195,0.03,392.0,2388.0,100.0,39.06,23.4190
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,21.61,553.75,2388.04,9044.07,1.3,47.49,522.28,2388.07,8131.49,8.4318,0.03,392.0,2388.0,100.0,39.00,23.4236
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,21.61,554.26,2388.08,9052.94,1.3,47.27,522.42,2388.03,8133.23,8.4178,0.03,390.0,2388.0,100.0,38.95,23.3442
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,21.61,554.45,2388.11,9049.48,1.3,47.13,522.86,2388.08,8133.83,8.3682,0.03,392.0,2388.0,100.0,38.88,23.3739
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,21.61,554.00,2388.06,9055.15,1.3,47.28,522.19,2388.04,8133.80,8.4294,0.03,393.0,2388.0,100.0,38.90,23.4044


In [4]:
display(pd.DataFrame({"dtype": train.dtypes.astype(str)}))

,dtype
unit_id,int64
cycle,int64
operational_setting_1,float64
operational_setting_2,float64
operational_setting_3,float64
sensor_1,float64
sensor_2,float64
sensor_3,float64
sensor_4,float64
sensor_5,float64


In [5]:
dataset_summary = pd.DataFrame(
    {"training": split_summary(train), "test_observed": split_summary(test)}
)
display(dataset_summary.round(2))
display(test_rul.head())
display(pd.DataFrame({"dtype": test_rul.dtypes.astype(str)}))
display(test_rul["rul_at_last_observation"].describe().to_frame().round(2))

,training,test_observed
rows,20631.00,13096.00
columns,26.00,26.00
engines,100.00,100.00
minimum_observed_cycles,128.00,31.00
median_observed_cycles,199.00,133.50
mean_observed_cycles,206.31,130.96
maximum_observed_cycles,362.00,303.00


,unit_id,rul_at_last_observation
0,1,112
1,2,98
2,3,69
3,4,82
4,5,91


,dtype
unit_id,int64
rul_at_last_observation,int64


,rul_at_last_observation
count,100.00
mean,75.52
std,41.76
min,7.00
25%,32.75
50%,86.00
75%,112.25
max,145.00


**Observation:** Training trajectories contain 20,631 engine-cycle rows; test trajectories contain 13,096 observed rows. Each split has 100 engines. Training trajectories are complete and therefore their maximum cycles are lifetimes; test maxima are only observed lengths. The RUL table has one row per test engine and is not joined to training data.

## Structural validation

The checks below report problems rather than repairing them. A malformed width, non-numeric token, non-finite value, or invalid identifier causes the loader to fail. These additional checks cover missing cells, duplicate observations, and broken per-engine cycle sequences.

In [6]:
validation = pd.DataFrame(
    {"training": validate_split(train), "test": validate_split(test)}
)
display(validation)
print(
    "Test RUL rows match test engines:",
    len(test_rul) == test["unit_id"].nunique(),
)

,training,test
missing_cells,0,0
duplicated_rows,0,0
duplicated_unit_cycle_pairs,0,0
non_finite_cells,0,0
non_positive_unit_ids,0,0
non_positive_cycles,0,0
engines_not_starting_at_cycle_1,0,0
engines_with_nonconsecutive_cycles,0,0


Test RUL rows match test engines: True


**Observation:** No missing cells, duplicate rows, duplicate unit-cycle keys, non-finite values, non-positive identifiers, or cycle-sequence gaps were found in FD001. The test RUL row count matches the number of test engines. This is evidence of structural consistency, not proof that every measurement is physically correct.

## Constant and near-constant columns

Here, a column is called near-constant when one exact value accounts for at least 95% of training rows while the column still has more than one value. The threshold is explicit because there is no universal definition of near-constant. No columns are removed.

In [7]:
profile = column_profile(train, near_constant_dominance=0.95)
constant_columns = profile.index[profile["is_constant"]].tolist()
near_constant_columns = profile.index[profile["is_near_constant"]].tolist()

print("Constant columns:", constant_columns)
print("Near-constant columns:", near_constant_columns)
display(
    profile.loc[constant_columns + near_constant_columns, [
        "unique_values", "variance", "dominant_value",
        "dominant_ratio", "is_constant", "is_near_constant"
    ]].round(6)
)
print("Observed operational-setting and sensor ranges:")
display(
    profile.loc[train.columns[2:], ["minimum", "maximum", "unique_values"]]
    .round(6)
)

Constant columns: ['operational_setting_3', 'sensor_1', 'sensor_5', 'sensor_10', 'sensor_16', 'sensor_18', 'sensor_19']
Near-constant columns: ['sensor_6']


,unique_values,variance,dominant_value,dominant_ratio,is_constant,is_near_constant
column,,,,,,
operational_setting_3,1,0.000000,100.00,1.000000,True,False
sensor_1,1,0.000000,518.67,1.000000,True,False
sensor_5,1,0.000000,14.62,1.000000,True,False
sensor_10,1,0.000000,1.30,1.000000,True,False
sensor_16,1,0.000000,0.03,1.000000,True,False
sensor_18,1,0.000000,2388.00,1.000000,True,False
sensor_19,1,0.000000,100.00,1.000000,True,False
sensor_6,2,0.000002,21.61,0.980321,False,True


Observed operational-setting and sensor ranges:


,minimum,maximum,unique_values
operational_setting_1,-0.0087,0.0087,158
operational_setting_2,-0.0006,0.0006,13
operational_setting_3,100.0000,100.0000,1
sensor_1,518.6700,518.6700,1
sensor_2,641.2100,644.5300,310
sensor_3,1571.0400,1616.9100,3012
sensor_4,1382.2500,1441.4900,4051
sensor_5,14.6200,14.6200,1
sensor_6,21.6000,21.6100,2
sensor_7,549.8500,556.0600,513


**Observation:** `operational_setting_3`, sensors 1, 5, 10, 16, 18, and 19 have exactly zero variance in FD001 training data. `sensor_6` has two values, but its dominant value occurs in roughly 98% of rows, so it meets the stated near-constant rule. These columns carry little or no within-FD001 variation. All observed measurement ranges are finite. Because the published text format does not supply hard validity bounds for each anonymized sensor, range inspection cannot prove physical correctness. Whether to remove any column is a later feature-design decision; this milestone only reports the evidence.

## Engine lifetimes

In [8]:
plot_engine_lifetimes(train, FIGURE_DIRECTORY / "fd001_engine_lifetimes.png")

PosixPath('/home/adnan/Documents/test/dev-lab/python/rul_predict/reports/figures/fd001_engine_lifetimes.png')

![FD001 engine lifetime distribution](../reports/figures/fd001_engine_lifetimes.png)

**Observation:** Complete training lifetimes range from 128 to 362 cycles. The median is 199 cycles and the mean is 206.31, with a longer upper tail caused by a small number of longer-lived engines.

**Interpretation:** Engines do not share a fixed lifetime, so trajectory length itself varies substantially across the fleet.

## Operational settings

In [9]:
selected_engines = representative_engine_ids(train)
print("Representative engines (short, median, long lifetime):", selected_engines)
display(train[["operational_setting_1", "operational_setting_2", "operational_setting_3"]].agg(["min", "max", "mean", "std", "nunique"]).round(6))
plot_operational_settings(train, FIGURE_DIRECTORY / "fd001_operational_settings.png")

Representative engines (short, median, long lifetime): (39, 26, 69)


,operational_setting_1,operational_setting_2,operational_setting_3
min,-0.008700,-0.000600,100.0
max,0.008700,0.000600,100.0
mean,-0.000009,0.000002,100.0
std,0.002187,0.000293,0.0
nunique,158.000000,13.000000,1.0


PosixPath('/home/adnan/Documents/test/dev-lab/python/rul_predict/reports/figures/fd001_operational_settings.png')

![FD001 operational settings](../reports/figures/fd001_operational_settings.png)

**Observation:** The third setting is exactly 100 in every training row. The first two settings fluctuate in narrow ranges around zero.

**Interpretation:** This agrees with FD001's single-condition design. Small numerical variation in settings 1 and 2 should not be confused with the six distinct operating regimes present in later C-MAPSS subsets.

## Sensor distributions, variance, and evolution

To select a compact set of sensors for display, we compute a Spearman rank correlation between each sensor and cycle separately within every training engine, then summarize those correlations across engines. This measures monotonic movement over time; it does not establish predictive value or causality.

In [10]:
trend_summary = sensor_trend_summary(train)
display(trend_summary.head(14).round(3))
display_sensors = trend_summary.dropna().head(6).index.tolist()
print("Sensors selected for compact distribution plots:", display_sensors)

,engines_with_variation,median_spearman_rho,q1_spearman_rho,q3_spearman_rho
sensor,,,,
sensor_11,100,0.819,0.780,0.848
sensor_12,100,-0.808,-0.833,-0.748
sensor_4,100,0.784,0.759,0.804
sensor_7,100,-0.774,-0.802,-0.715
sensor_8,100,0.768,0.603,0.816
sensor_13,100,0.762,0.606,0.821
sensor_9,100,0.739,-0.193,0.911
sensor_15,100,0.721,0.675,0.746
sensor_21,100,-0.708,-0.750,-0.658


Sensors selected for compact distribution plots: ['sensor_11', 'sensor_12', 'sensor_4', 'sensor_7', 'sensor_8', 'sensor_13']


In [11]:
plot_sensor_distributions(
    train, display_sensors, FIGURE_DIRECTORY / "fd001_sensor_distributions.png"
)
plot_sensor_variance(train, FIGURE_DIRECTORY / "fd001_sensor_variance.png")
plot_sensor_evolution(
    train, display_sensors[:4], FIGURE_DIRECTORY / "fd001_sensor_evolution.png"
)
plot_sensor_trends(trend_summary, FIGURE_DIRECTORY / "fd001_sensor_trends.png")

PosixPath('/home/adnan/Documents/test/dev-lab/python/rul_predict/reports/figures/fd001_sensor_trends.png')

![Selected FD001 sensor distributions](../reports/figures/fd001_sensor_distributions.png)

![FD001 sensor variance](../reports/figures/fd001_sensor_variance.png)

![Selected FD001 sensor evolution](../reports/figures/fd001_sensor_evolution.png)

![FD001 within-engine sensor trends](../reports/figures/fd001_sensor_trends.png)

**Observations:** Sensors 11 and 4 tend to rise with cycle, while sensors 12 and 7 tend to fall; their median within-engine rank correlations have large magnitude and consistent interquartile ranges. Sensors 8 and 13 also usually rise. Sensors 9 and 14 have positive median trends but vary much more between engines. The raw sensor distributions use different physical units and scales.

**Interpretation:** Several channels visibly respond as degradation progresses, but cycle association alone does not prove that a channel will improve unseen-engine RUL prediction. Raw variance magnitudes cannot be compared as feature importance because sensor units differ.

## Sensor correlation

In [12]:
nonconstant_sensors = [
    sensor for sensor in SENSOR_COLUMNS if train[sensor].nunique() > 1
]
sensor_correlation = train[nonconstant_sensors].corr()
upper_triangle = sensor_correlation.where(
    np.triu(np.ones(sensor_correlation.shape), k=1).astype(bool)
)
strongest_pairs = (
    upper_triangle.stack()
    .rename("pearson_correlation")
    .reset_index()
)
strongest_pairs.columns = ["sensor_a", "sensor_b", "pearson_correlation"]
strongest_pairs = strongest_pairs.iloc[
    strongest_pairs["pearson_correlation"].abs().argsort()[::-1]
].head(10)
display(strongest_pairs.round(3))
plot_sensor_correlation(train, FIGURE_DIRECTORY / "fd001_sensor_correlation.png")

,sensor_a,sensor_b,pearson_correlation
224,sensor_21,sensor_21,NaN
223,sensor_21,sensor_20,NaN
222,sensor_21,sensor_17,NaN
221,sensor_21,sensor_15,NaN
220,sensor_21,sensor_14,NaN
219,sensor_21,sensor_13,NaN
218,sensor_21,sensor_12,NaN
217,sensor_21,sensor_11,NaN
216,sensor_21,sensor_9,NaN
215,sensor_21,sensor_8,NaN


PosixPath('/home/adnan/Documents/test/dev-lab/python/rul_predict/reports/figures/fd001_sensor_correlation.png')

![FD001 sensor correlation](../reports/figures/fd001_sensor_correlation.png)

**Observation:** Sensor 9 and sensor 14 have the strongest linear relationship (Pearson correlation about 0.963). Several other sensor pairs also show strong positive or negative correlations.

**Interpretation:** Correlated sensors may respond to related engine behavior or shared degradation, but correlation does not make them interchangeable and does not justify dropping either one at this stage.

## Milestone 1 conclusions and limits

FD001 is structurally clean, but it is not uniformly informative: seven measurement/setting columns are constant and one sensor is near-constant under an explicit 95% dominance rule. Complete training lifetimes vary widely, and multiple non-constant sensors show consistent monotonic movement within engines. Strong sensor-to-sensor correlation is also present.

These are descriptive findings. This notebook does not test generalization, quantify predictive utility, choose model features, create RUL targets, or use the supplied test RUL values as inputs. Those decisions require later milestones.